# Add the MPC designation to the Rubin observations
#### *Open Tools, Open Sky: Doing LSST Science with Public Rubin Data & LINCC Frameworks, UdeA, Medellín, October 13 2026*

Each Rubin detection submitted to the Minor Planet Center is identified by its `permid` (number) and/or `provid`
(provisional designation). Here we find, for every detection, the readable designation of the object in the MPC
orbit catalogue (MPCORB `designation`, e.g. `(1) Ceres`, `(800001) 2013 YW88` or `2021 AQ`) and save it as a new
column, `mpc_designation`. Afterwards the orbits can be added to the observations with a simple join on the designation
(see `exploration_data.ipynb`).

In [36]:
from util import *
import pandas as pd
import numpy as np

# Directories
data_dir = 'data/'

# Check that the data files exist (download them otherwise)
get_data(data_dir=data_dir, only=['obs_sbn_X05_full.parquet', 'mpcorb_extended.dat.gz'])

Data already exists: data/obs_sbn_X05_full.parquet
Data already exists: data/mpcorb_extended.dat.gz
Download complete or files already present.


In [37]:
# Vera C. Rubin Observatory observations submitted to the Minor Planet Center
data = pd.read_parquet(data_dir + 'obs_sbn_X05_full.parquet', engine='pyarrow')
print(f'{len(data):,} detections')

8,084,389 detections


## MPCORB designations

We only need the designation fields of `data/mpcorb_extended.dat.gz` (same fixed-width format as in `exploration_data.ipynb`):
the readable `designation` and, from the extended version, the `other_desigs` of each object.

In [38]:
import gzip

path = data_dir + 'mpcorb_extended (1).dat.gz'
colspecs = {
    'designation':  (166, 194),   # readable designation, e.g. '(1) Ceres' or '2021 AQ'
    'other_desigs': (217, 400),   # extended: other designations of the same object
}

# The data start after the header's line of dashes
with gzip.open(path, 'rt') as f:
    for i, line in enumerate(f):
        if line.startswith('-----'):
            header_lines = i + 1
            break

mpcorb = pd.read_fwf(path, compression='gzip', skiprows=header_lines, header=None,
                     colspecs=list(colspecs.values()), names=list(colspecs.keys()),
                     skip_blank_lines=True, dtype_backend='pyarrow')
print(f'{len(mpcorb):,} objects, designations unique: {mpcorb["designation"].is_unique}')

1,576,448 objects, designations unique: True


In [39]:
mpcorb[mpcorb['designation'] == '2007 CL70']

,designation,other_desigs
922441,2007 CL70,2014 WT190 2025 MM27


In [40]:
# Identifiers in the same form as the observation file, so the two can be matched:
# '(1) Ceres' -> permid '1'; '(800001) 2013 YW88' -> permid '800001', provid '2013 YW88'; '2021 AQ' -> provid '2021 AQ'
numbered = mpcorb['designation'].str.extract(r'^\((?P<permid>\d+)\)\s*(?P<rest>.*)$')
rest_is_provid = numbered['rest'].str.match(r'^\d{4} [A-Z]{2}\d*$') == True
mpcorb['permid'] = numbered['permid']
mpcorb['provid'] = mpcorb['designation'].where(mpcorb['permid'].isna(), numbered['rest'].where(rest_is_provid))

## Join the two catalogues

An object in MPCORB can be found by three keys: its number (`permid`), its main provisional designation (`provid`),
or one of its other designations (`other_desigs`). We make one small table `key -> designation` for each key and
join it to the observations with `merge`. The joins are done once per distinct (`permid`, `provid`) pair, not for all
8M detections. Priority: `permid`, then `provid`, then the other designations; `match_by` records which one matched.
Detections with both `permid` and `provid` null (and comets, which are not in MPCORB) get no designation.

In [41]:
# 1. One table per key: key -> MPC designation
by_permid = mpcorb[['permid', 'designation']].dropna()
by_provid = mpcorb[['provid', 'designation']].dropna()

by_other = mpcorb[['designation', 'other_desigs']].dropna()
by_other['provid'] = by_other['other_desigs'].str.strip().str.split(r'\s{2,}', regex=True)
by_other = by_other.explode('provid')[['provid', 'designation']].dropna()

# A designation should point to one object: drop the (rare) ambiguous ones
by_provid = by_provid.drop_duplicates('provid', keep=False)
by_other = by_other.drop_duplicates('provid', keep=False)

# 2. Join each table to the distinct (permid, provid) pairs of the observations
pairs = data[['permid', 'provid']].drop_duplicates()
pairs = pairs.merge(by_permid.rename(columns={'designation': 'desig_permid'}), on='permid', how='left')
pairs = pairs.merge(by_provid.rename(columns={'designation': 'desig_provid'}), on='provid', how='left')
pairs = pairs.merge(by_other.rename(columns={'designation': 'desig_other'}), on='provid', how='left')

# 3. Keep the first match in order of priority
pairs['mpc_designation'] = pairs['desig_permid'].fillna(pairs['desig_provid']).fillna(pairs['desig_other'])
pairs['match_by'] = np.select([pairs['desig_permid'].notna(), pairs['desig_provid'].notna(), pairs['desig_other'].notna()],
                              ['permid', 'provid', 'other_desig'], default=None)

# 4. Bring the result back to every detection (a left merge keeps the row order;
#    pandas also matches null keys to null keys, so rows with a null permid or provid are handled)
matched = data[['permid', 'provid']].merge(pairs[['permid', 'provid', 'mpc_designation', 'match_by']],
                                           on=['permid', 'provid'], how='left')
data['mpc_designation'] = matched['mpc_designation'].array
data['match_by'] = matched['match_by'].to_numpy()

has_desig = data['mpc_designation'].notna()
print(f"Detections with an MPC designation: {has_desig.sum():,} of {len(data):,} ({100 * has_desig.mean():.2f}%)")
data['match_by'].value_counts(dropna=False)

Detections with an MPC designation: 7,961,702 of 8,084,389 (98.48%)


match_by
permid         4787767
provid         3155203
None            122687
other_desig      18732
Name: count, dtype: int64

## Objects not in MPCORB: ask the MPC

Some objects observed by Rubin are not in our copy of MPCORB, mostly because the MPC linked their new Rubin
designation to an older object after the file was downloaded (e.g. `2026 DO29` is now `2015 CY86`).
For these we ask the MPC Designation Identifier API (`https://data.minorplanetcenter.net/api/query-identifier`),
which returns the current number, name and primary designation of each object. If MPCORB has the object under that
number or designation we use its MPCORB `designation`; otherwise we build it in the same format. Comets, which are
not in MPCORB, keep the IAU designation given by the MPC. These detections get `match_by = 'mpc_api'`.
The answers are cached in `data/mpc_query_identifier.json`, so the API is only queried once.

In [42]:
import json, os
import requests

# 1. Distinct objects of the observations that were not found in MPCORB.
#    We ask the MPC by the provisional designation, or by the number when there is none.
missing = pairs.loc[pairs['mpc_designation'].isna(), ['permid', 'provid']].copy()
missing['query_id'] = missing['provid'].fillna(missing['permid'])
missing = missing.dropna(subset=['query_id'])
ids = sorted(missing['query_id'].unique())
print(f'{len(ids):,} objects not in MPCORB')

# 2. Ask the MPC Designation Identifier API (in batches of 100); the answers are saved
#    in a cache file so the queries are only made once
cache_path = data_dir + 'mpc_query_identifier.json'
cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
to_query = [i for i in ids if i not in cache]
for start in range(0, len(to_query), 100):
    batch = to_query[start:start + 100]
    r = requests.get('https://data.minorplanetcenter.net/api/query-identifier',
                     json={'ids': batch}, timeout=120)
    r.raise_for_status()
    cache.update(r.json())
    print(f'  queried {start + len(batch):,} of {len(to_query):,}', end='\r')
json.dump(cache, open(cache_path, 'w'))

# 3. MPC designation in the MPCORB format: '(number) Name', '(number) provid' or 'provid'
#    (comets keep the IAU designation given by the MPC, e.g. 'C/2025 L1')
mpcorb_by_permid = dict(zip(by_permid['permid'], by_permid['designation']))
mpcorb_by_provid = dict(zip(by_provid['provid'], by_provid['designation']))

def api_designation(ans):
    if not ans or ans.get('found') != 1:
        return None
    if ans['object_type'][0] != 'Minor Planet':
        return ans['iau_designation']
    permid, provid = ans['permid'], ans['unpacked_primary_provisional_designation']
    # if MPCORB has the object under its current number or primary designation, use that one
    if permid in mpcorb_by_permid:
        return mpcorb_by_permid[permid]
    if provid in mpcorb_by_provid:
        return mpcorb_by_provid[provid]
    if permid is not None:
        return f"({permid}) {ans['name'] or provid}"
    return provid

missing['api_designation'] = missing['query_id'].map(lambda i: api_designation(cache.get(i)))
found = missing.dropna(subset=['api_designation'])
print(f'\nFound by the MPC API: {len(found):,} of {len(ids):,} objects')

# 4. Add them to the observations (same null-safe merge as above)
api = data[['permid', 'provid']].merge(found[['permid', 'provid', 'api_designation']],
                                       on=['permid', 'provid'], how='left')['api_designation']
new = data['mpc_designation'].isna() & api.notna().to_numpy()
data.loc[new, 'mpc_designation'] = api[new].to_numpy()
data.loc[new, 'match_by'] = 'mpc_api'

has_desig = data['mpc_designation'].notna()
print(f"Detections with an MPC designation: {has_desig.sum():,} of {len(data):,} ({100 * has_desig.mean():.2f}%)")
data['match_by'].value_counts(dropna=False)

2,438 objects not in MPCORB

Found by the MPC API: 2,438 of 2,438 objects
Detections with an MPC designation: 8,022,026 of 8,084,389 (99.23%)


match_by
permid         4787767
provid         3155203
None             62363
mpc_api          60324
other_desig      18732
Name: count, dtype: int64

## Save the Rubin catalogue with the new columns

We also add `n_provids`, the number of different Rubin provids of each object (same `mpc_designation`):
2 means Rubin reported the object as two separate objects (e.g. `2025 JQ7` and `2025 NF311` are both `2025 NF311`),
0 that it was reported only by its number. `write_parquet` (from `parquet_docs.py`) saves the description of each
column inside the file; read them with `column_descriptions(path)`.

In [43]:
from parquet_docs import write_parquet

# Number of different Rubin provids of each object (null if the detection has no mpc_designation)
data['n_provids'] = data.groupby('mpc_designation')['provid'].transform('nunique').astype('Int64')
print(data.drop_duplicates('mpc_designation')['n_provids'].value_counts().rename('objects'))

out = data_dir + 'obs_sbn_X05_full_mpc_desig.parquet'
write_parquet(data, out)
print(f'Saved {len(data):,} rows x {data.shape[1]} columns to {out}')

n_provids
1    295280
2      2989
0       117
Name: objects, dtype: Int64
Saved 8,084,389 rows x 93 columns to data/obs_sbn_X05_full_mpc_desig.parquet


In [48]:
# Description of the file and of each column (saved in the parquet schema; pandas does not show them)
from parquet_docs import column_descriptions
pd.set_option('display.max_colwidth', None)
column_descriptions(data_dir + 'obs_sbn_X05_full_mpc_desig.parquet')

Rubin (X05) observations submitted to the Minor Planet Center (MPC obs_sbn table, one row per detection), with the MPC designation of the object of each detection (mpc_designation, match_by, n_provids). Written by add_mpc_designation.ipynb.


,column,type,description
0,id,int64,Row id of the observation in the MPC observations table (obs_sbn).
1,trksub,string,Tracklet id given by the submitter (Rubin): detections of the same object in one night share it.
2,trkid,string,Tracklet id assigned by the MPC.
3,obsid,string,Unique observation id assigned by the MPC.
4,submission_id,string,Id of the MPC submission (batch of observations) that contained the observation.
5,submission_block_id,string,"Id of the block, inside the MPC submission, that contained the observation."
6,obs80,string,The observation in the old MPC 80-column format.
7,status,string,MPC processing status code of the observation.
8,ref,string,Publication reference of the observation (e.g. 'MPEC 2026-Q94'); null if not published yet.
9,healpix,int64,HEALPix pixel index of the sky position (computed by the MPC).


## Table of equivalences `provid -> mpc_designation`

`data/mpc_designations.csv` has one row per Rubin `provid` with the MPC object it belongs to. One MPC object can have
several Rubin provids (e.g. `2025 JQ7` and `2025 NF311` are both `2025 NF311`), so we add two columns:

- `is_primary`: the provid is the main provisional designation of the object according to the MPC
- `n_provids`: number of Rubin provids of the object (2 means Rubin reported the object as two separate ones)

The main provisional designation comes from MPCORB:
- `2014 HR48` -> `2014 HR48` (unnumbered: the designation itself)
- `(831800) 2009 WC150` -> `2009 WC150` (numbered: the designation after the number)
- `(22754) Olympus` -> `1998 WJ8` (numbered and named: the first of the other designations in MPCORB extended)

For objects not in MPCORB (e.g. comets), it comes from the MPC API answers.
An object can have no primary provid in Rubin (e.g. `2025 HB67`, whose main designation `2022 UK169` was not observed by Rubin).

In [44]:
# 1. Main provisional designation of each MPC object
provid_pattern = r'^\d{4} ([A-Z]{2}\d*|P-L|T-[123])$'   # e.g. '2009 WC150', or the survey ones '2084 P-L', '2058 T-2'
rest = mpcorb['designation'].str.replace(r'^\(\d+\)\s*', '', regex=True)
named = mpcorb['designation'].str.startswith('(') & ~rest.str.match(provid_pattern).fillna(False).astype(bool)
# first of the other designations (they are separated by 2+ spaces)
first_other = mpcorb['other_desigs'].str.extract(r'^\s*(?P<first>.+?)(?:\s{2,}|\s*$)')['first']
primary = dict(zip(mpcorb['designation'], rest.where(~named, first_other)))

def primary_provid(provid, mpc_designation):
    if mpc_designation in primary:
        return primary[mpc_designation]
    return (cache.get(provid) or {}).get('unpacked_primary_provisional_designation')   # not in MPCORB: MPC API

# 2. Every (provid, mpc_designation) pair of the Rubin catalogue, with the two new columns
rubin_ids = data[['provid', 'mpc_designation']].dropna().drop_duplicates()
rubin_ids['is_primary'] = [p == primary_provid(p, d) for p, d in zip(rubin_ids['provid'], rubin_ids['mpc_designation'])]
rubin_ids['n_provids'] = rubin_ids.groupby('mpc_designation')['provid'].transform('nunique')

# 3. Add them to the table of equivalences (its rows keep their order; Rubin provids missing from it go at the end).
#    Only the two id columns are read, so the cell also works when the file already has is_primary and n_provids
mpc_design = pd.read_csv(data_dir + 'mpc_designations.csv', usecols=['provid', 'mpc_designation'])
extra = rubin_ids.loc[~rubin_ids['provid'].isin(mpc_design['provid']), ['provid', 'mpc_designation']]
mpc_design = pd.concat([mpc_design, extra], ignore_index=True)
mpc_design = mpc_design.merge(rubin_ids[['provid', 'is_primary', 'n_provids']], on='provid', how='left')

print(f'{len(mpc_design):,} provids ({len(extra):,} added), {mpc_design["is_primary"].sum():,} primary')
print(mpc_design['n_provids'].value_counts().rename('provids per object'))
examples = ['2014 HR48', '2025 NF311', '(831800) 2009 WC150', '(22754) Olympus', '2022 UK169', '2002 PG143']
mpc_design[mpc_design['mpc_designation'].isin(examples)].sort_values(['mpc_designation', 'is_primary'], ascending=[True, False])

301,258 provids (0 added), 295,599 primary
n_provids
1    295280
2      5978
Name: provids per object, dtype: int64


,provid,mpc_designation,is_primary,n_provids
35602,1998 WJ8,(22754) Olympus,True,1
129631,2009 WC150,(831800) 2009 WC150,True,2
646,2025 MN83,(831800) 2009 WC150,False,2
57237,2025 PJ255,2002 PG143,False,2
59138,2025 NX412,2002 PG143,False,2
129106,2014 HR48,2014 HR48,True,2
246,2025 MD38,2014 HR48,False,2
31055,2025 HB67,2022 UK169,False,1
42617,2025 NF311,2025 NF311,True,2
45562,2025 JQ7,2025 NF311,False,2


In [46]:
out = data_dir + 'mpc_designations.csv'
mpc_design.to_csv(out, index=False)
print(f'Saved {len(mpc_design):,} rows x {mpc_design.shape[1]} columns to {out}')

Saved 301,258 rows x 4 columns to data/mpc_designations.csv
